# 03. Combinations

**Tier:** Deep. **Why you'd use this:** You need how many unordered ways coins can sum to an amount. Order of coins in a way should not matter; `[1, 2]` and `[2, 1]` are one combination.

**After this lesson you can:** fill a ways table with coins on the outer loop; see why flipping the loops counts permutations instead.
About 25 minutes. **Uses:** [Circle](02-circle-lesson.ipynb), [Memoization](../../course/09-recursion/04-memoization-lesson.ipynb)


## Quick review

1. What does `ways[total]` store in a combinations table?
2. Why put the coin loop outside the totals loop?

<details><summary>Answers</summary>

How many unordered combinations sum to that total. Outer coins means each combination is built by adding one coin type at a time, so order of addition is fixed by coin order and permutations are not double-counted.
</details>


## Recognize it

The prompt asks for the number of combinations that make a target, often with unlimited uses of each coin or item. Wording sounds like "number of ways", "combinations", or "order does not matter".

Signals: a target amount, a list of denominations, and a count rather than the fewest coins. A look-alike that is not this pattern: fewest coins to make the amount — that is a minimum DP, not a counting DP.

If order matters — sequences rather than bags — the nested loop order flips and you are counting permutations. Read the prompt for "combinations" versus "permutations" before you lock the loops.


## Why it works this way

Let `ways[total]` be how many unordered combinations of the given coins sum to `total`. Start with `ways[0] = 1`: one way to make nothing, use no coins. For each coin, walk totals from that coin up to the amount and add `ways[total - coin]` into `ways[total]`. Reading the recurrence: every combination for `total` is a combination for a smaller total plus one more of this coin.

Loop order is the design. Coins on the outside means you finish considering coin `c` before you touch the next coin. A combination is then counted once, in increasing coin-type order. Totals on the outside with coins inside counts sequences: `[1, 2]` and `[2, 1]` both get added, which is permutations, not combinations. The same table shape with a `min` update instead of `+=` solves fewest coins; the recurrence family is shared, the aggregation is not.

A recursive search that tries every multiset also works and can memoize on `(index, remaining)`, but without memory it re-explores the same remaining amount from the same coin index. The bottom-up table solves each pair once. Time is proportional to `amount * len(coins)`. Extra memory is one integer per total from `0` through `amount`.

See [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) for why a flat `ways` list is a cheap random-access table, and [Memoization](../../course/09-recursion/04-memoization-lesson.ipynb) for the top-down twin. Greedy largest-coin-first fails when denominations are awkward; counting needs every combination, not one path.

The base case `ways[0] = 1` is easy to forget and silently zeros the whole table. Without it, no later total can grow from a valid empty combination. Treat that seed as part of the recurrence, not as a decorative default.


## When to use it

Use a combinations table when the answer is how many unordered ways to hit a target with reusable parts. Prefer fewest-coin DP when the answer is a minimum number of parts. Prefer permutation counting (totals outer) only when order matters. Prefer backtracking when you must list the combinations, not only count them. Prefer a closed formula when the parts are trivial, such as only `1`-coins.

If each item may be used at most once, walk totals downward for that item (0/1 knapsack style) instead of upward. Unlimited reuse is what makes the upward inner walk safe here.


## Brute force, and why it's slow

Count recursive calls that try each coin for the remaining amount without memory.


In [1]:
def combinations_slow(coins, amount):
    calls = 0

    def ways(remaining, start):
        nonlocal calls
        calls += 1
        if remaining == 0:
            return 1
        total = 0
        for i in range(start, len(coins)):
            coin = coins[i]
            if coin <= remaining:
                total += ways(remaining - coin, i)
        return total

    answer = ways(amount, 0)
    return answer, calls


for amount in (8, 12, 16):
    print(amount, combinations_slow([1, 2, 5], amount))


8 (7, 31)
12 (13, 73)
16 (20, 141)


## Key insight

`ways[0] = 1`. For each coin, add `ways[total - coin]` into later totals. Outer coin loop counts each multiset once; swapping the loops counts ordered sequences instead.


## Worked example

Given coin denominations and an amount, return how many unordered combinations sum to that amount. Each coin may be used any number of times.


In [2]:
def count_combinations(coins: list[int], amount: int) -> int:
    # 1. ways[total] = unordered combinations that sum to total.
    ways = [0] * (amount + 1)
    ways[0] = 1
    # 2. Outer coins: each combination counted once.
    for coin in coins:
        for total in range(coin, amount + 1):
            # 3. Add every way to make the remainder.
            ways[total] += ways[total - coin]
    return ways[amount]


print(count_combinations([1, 2, 5], 5))
print(count_combinations([2], 3))
print(count_combinations([1, 2, 5], 0))


4
0
1


## Trace

Coins `[1, 2, 5]`, amount `5`. After each coin, `ways[0..5]`:

| after coin | ways |
|---|---|
| start | `[1, 0, 0, 0, 0, 0]` |
| `1` | `[1, 1, 1, 1, 1, 1]` |
| `2` | `[1, 1, 2, 2, 3, 3]` |
| `5` | `[1, 1, 2, 2, 3, 4]` |

Answer **4**: `{5}`, `{2,2,1}`, `{2,1,1,1}`, `{1,1,1,1,1}`.


## Cost

One nested loop over coins and totals: time grows with `amount * len(coins)`. The table holds `amount + 1` integers. That beats the unmemoized recursion because each remaining total is updated a bounded number of times. Listing combinations with backtracking can emit exponentially many answers even when the count fits in one integer.


## Variants

Not solved here: [Coin Change](../../challenges/problems/19-coin-change.ipynb) (fewest coins), [Combination Sum](../../challenges/problems/34-combination-sum.ipynb) (list the combinations), [Unique Paths](../../challenges/problems/37-unique-paths.ipynb).


## Common mistake

This bug is silent. The loops are flipped: totals outer, coins inner. That counts permutations. For `[1, 2, 5]` and amount `5` the print is `9`, and the combination count is `4`.


In [3]:
def count_combinations_wrong(coins, amount):
    ways = [0] * (amount + 1)
    ways[0] = 1
    for total in range(1, amount + 1):  # totals outer → permutations
        for coin in coins:
            if coin <= total:
                ways[total] += ways[total - coin]
    return ways[amount]


print(count_combinations_wrong([1, 2, 5], 5))


9


The fix puts coins on the outside. The buggy cell prints `9` for coins `[1, 2, 5]` and amount `5` because order of loops counts permutations.


In [4]:
print(count_combinations([1, 2, 5], 5))


4


## Python idioms

A flat list indexed by total is enough; you do not need a matrix. `functools.cache` on `(index, remaining)` is the top-down form with coins ordered by index.


In [5]:
import functools


def count_combinations_cached(coins: list[int], amount: int) -> int:
    @functools.cache
    def ways(index: int, remaining: int) -> int:
        if remaining == 0:
            return 1
        if index == len(coins) or remaining < 0:
            return 0
        return ways(index, remaining - coins[index]) + ways(index + 1, remaining)

    return ways(0, amount)


print(count_combinations_cached([1, 2, 5], 5))


4


## Recap

- `ways[0] = 1`; larger totals add from `total - coin`.
- Coins outer counts combinations; totals outer counts permutations.
- Fewest coins uses the same shape with `min`, not `+=`.

Next: [practice](03-combinations-practice.ipynb).
